<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# The Linear Regression Engine
## Dataset: Silicon Fmax Prediction (Post-Silicon Validation)

**Project:** 001 — The AI Engineering Lab  
**Objective:** Apply the same linear regression pipeline to a post-silicon validation use case — predicting the maximum stable clock frequency (Fmax in MHz) of a chip across voltage and temperature corners.

**Engineering Context:**  
In post-silicon validation, engineers characterize silicon across voltage (`vdd_core`), temperature (`junction_temp`), and process corners to determine the maximum frequency at which a chip can operate reliably. Traditionally this requires exhaustive corner sweeps. A regression model trained on measured data can predict Fmax for untested corners, reducing characterization time significantly.

**Dataset:** Synthetic silicon characterization data with 2,000 observations across voltage, temperature, leakage, and process monitor readings.

---
## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

Libraries loaded.


---
## 2. Load and Explore the Fmax Dataset

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
df = pd.read_csv('dataset_a.csv')  # Ensure the file is in the current directory or provide the correct path
print(f'Shape: {df.shape}')
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_a.csv'

In [ ]:
df.describe().round(3)

### 2.1 EDA: Voltage and Temperature vs Fmax

The most important relationships in silicon characterization are VDD vs Fmax (positive) and temperature vs Fmax (negative). Let us visualize these directly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

features = ['vdd_core', 'junction_temp', 'leakage_current', 'ring_oscillator_speed', 'ir_drop_estimate', 'thermal_resistance']
colors   = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']

for ax, feat, color in zip(axes.flat, features, colors):
    ax.scatter(df[feat], df['fmax_mhz'], alpha=0.2, s=8, color=color)
    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('Fmax (MHz)', fontsize=10)
    ax.set_title(f'{feat} vs Fmax', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature vs Target Relationships — Silicon Fmax Dataset', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../assets/fig_fmax_eda.png', bbox_inches='tight')
plt.show()

---
## 3. Preprocessing and Splitting

In [ ]:
TARGET = 'fmax_mhz'
NUMERIC_FEATURES = ['vdd_core', 'junction_temp', 'leakage_current',
                    'ring_oscillator_speed', 'thermal_resistance', 'ir_drop_estimate']
CATEGORICAL_FEATURES = ['silicon_lot_id']

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), CATEGORICAL_FEATURES)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f'Train: {X_train_proc.shape} | Val: {X_val_proc.shape} | Test: {X_test_proc.shape}')

---
## 4. Model Training and Comparison

In [ ]:
alphas_cv = np.logspace(-3, 3, 100)

ols   = LinearRegression().fit(X_train_proc, y_train)
ridge = RidgeCV(alphas=alphas_cv, cv=5).fit(X_train_proc, y_train)
lasso = LassoCV(alphas=alphas_cv, cv=5, max_iter=10000).fit(X_train_proc, y_train)
enet  = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_train_proc, y_train)

models = {'OLS': ols, 'Ridge': ridge, 'Lasso': lasso, 'Elastic Net': enet}

print(f'{"Model":<15} {"Val RMSE":>10} {"Val MAE":>10} {"Val R2":>10}')
print('-' * 48)
for name, model in models.items():
    y_pred = model.predict(X_val_proc)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae  = mean_absolute_error(y_val, y_pred)
    r2   = r2_score(y_val, y_pred)
    print(f'{name:<15} {rmse:>10.2f} {mae:>10.2f} {r2:>10.4f}')

print(f'\nBest Ridge alpha: {ridge.alpha_:.4f}')
print(f'Best Lasso alpha: {lasso.alpha_:.4f}')
print(f'Lasso non-zero features: {np.sum(lasso.coef_ != 0)} / {len(lasso.coef_)}')

---
## 5. Signature 3D Visualization: Cost Function Surface

The MSE cost function for linear regression forms a **convex bowl** in coefficient space. Gradient Descent follows the slope of this bowl downward to the minimum. Below we visualize this surface for two selected coefficients (VDD and junction temperature) while holding all others at their optimal values.

In [ ]:
# Use the two most important features for the 3D surface
# Train a simple 2-feature model for visualization clarity
X_2feat = X_train_proc[:, :2]  # vdd_core and junction_temp (after scaling)
y_train_arr = y_train.values

# Compute cost over a grid of (w0, w1) values
w0_range = np.linspace(-200, 2000, 60)
w1_range = np.linspace(-200, 200, 60)
W0, W1 = np.meshgrid(w0_range, w1_range)

# Optimal weights for 2-feature model
ols_2f = LinearRegression(fit_intercept=True).fit(X_2feat, y_train_arr)
w0_opt, w1_opt = ols_2f.coef_[0], ols_2f.coef_[1]

COST = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        y_hat = X_2feat @ np.array([W0[i,j], W1[i,j]]) + ols_2f.intercept_
        COST[i,j] = np.mean((y_train_arr - y_hat)**2)

# Simulate gradient descent path
lr = 0.08
w = np.array([-150.0, -150.0])
path = [w.copy()]
for _ in range(60):
    y_hat = X_2feat @ w + ols_2f.intercept_
    grad = -2 * X_2feat.T @ (y_train_arr - y_hat) / len(y_train_arr)
    w = w - lr * grad
    path.append(w.copy())
path = np.array(path)

# Compute cost along path
path_cost = []
for p in path:
    y_hat = X_2feat @ p + ols_2f.intercept_
    path_cost.append(np.mean((y_train_arr - y_hat)**2))

fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(W0, W1, COST, cmap='viridis', alpha=0.6, linewidth=0)
ax.plot(path[:,0], path[:,1], path_cost, 'r.-', markersize=5, linewidth=2, label='Gradient Descent Path')
ax.scatter([w0_opt], [w1_opt], [np.min(COST)], color='gold', s=100, zorder=5, label='Global Minimum')

ax.set_xlabel('w₀ (VDD coefficient)', fontsize=10, labelpad=10)
ax.set_ylabel('w₁ (Temp coefficient)', fontsize=10, labelpad=10)
ax.set_zlabel('MSE Cost', fontsize=10, labelpad=10)
ax.set_title('3D MSE Cost Surface with Gradient Descent Path\n(Silicon Fmax Prediction)', 
             fontsize=13, fontweight='bold', pad=15)
ax.legend(loc='upper right', fontsize=9)
fig.colorbar(surf, ax=ax, shrink=0.4, aspect=10, label='MSE')

plt.tight_layout()
plt.savefig('../assets/fig1_cost_surface_3d.png', bbox_inches='tight', dpi=150)
plt.show()
print('Signature 3D visualization saved.')

---
## 6. L1 vs L2 Regularization Contour Plot

The geometric interpretation of regularization explains why Lasso creates sparsity. The OLS solution is the center of the elliptical contours. Regularization constrains the solution to lie within a region: a **sphere** for Ridge (L2) and a **diamond** for Lasso (L1). The diamond has corners on the axes, so the constrained solution is more likely to land exactly on an axis — meaning one coefficient becomes exactly zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# OLS minimum (simplified 2D for illustration)
w_ols = np.array([1.6, 0.8])

# Cost function contours (ellipses centered at OLS minimum)
w1_grid = np.linspace(-2, 2.5, 300)
w2_grid = np.linspace(-1.5, 2, 300)
W1g, W2g = np.meshgrid(w1_grid, w2_grid)
# Elliptical cost (scaled to look realistic)
COST_GRID = 0.8*(W1g - w_ols[0])**2 + 2.5*(W2g - w_ols[1])**2

for ax, title, constraint_label, color in zip(
    axes,
    ['Ridge (L2): Sphere Constraint', 'Lasso (L1): Diamond Constraint'],
    ['L2 Ball: w₁² + w₂² ≤ t', 'L1 Diamond: |w₁| + |w₂| ≤ t'],
    ['#4CAF50', '#FF9800']
):
    ax.contour(W1g, W2g, COST_GRID, levels=12, cmap='Blues', alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('w₁ (VDD coefficient)')
    ax.set_ylabel('w₂ (Temperature coefficient)')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.scatter(*w_ols, color='blue', s=80, zorder=5, label='OLS Minimum')
    ax.text(w_ols[0]+0.05, w_ols[1]+0.05, 'OLS Min', fontsize=9, color='blue')
    ax.text(-1.8, -1.2, constraint_label, fontsize=9, color=color,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Draw L2 sphere
theta = np.linspace(0, 2*np.pi, 200)
r = 0.9
axes[0].plot(r*np.cos(theta), r*np.sin(theta), color='#4CAF50', linewidth=2.5, label='L2 Constraint')
axes[0].scatter([0.87], [0.22], color='#4CAF50', s=100, zorder=6, label='Constrained Solution')
axes[0].legend(fontsize=9)

# Draw L1 diamond
r = 0.9
diamond_x = [r, 0, -r, 0, r]
diamond_y = [0, r, 0, -r, 0]
axes[1].plot(diamond_x, diamond_y, color='#FF9800', linewidth=2.5, label='L1 Constraint')
axes[1].scatter([0.0], [0.9], color='#FF9800', s=100, zorder=6, label='Constrained Solution (on axis!)')
axes[1].legend(fontsize=9)

plt.suptitle('Why Lasso Creates Sparsity: Geometric Interpretation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/fig2_l1_l2_contours.png', bbox_inches='tight', dpi=150)
plt.show()
print('L1 vs L2 contour plot saved.')

---
## 7. Final Test Set Evaluation

In [ ]:
print('Final Test Set Results — Silicon Fmax Prediction')
print(f'{"Model":<15} {"RMSE (MHz)":>12} {"MAE (MHz)":>12} {"R2":>10}')
print('-' * 52)
for name, model in models.items():
    y_pred = model.predict(X_test_proc)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    print(f'{name:<15} {rmse:>12.2f} {mae:>12.2f} {r2:>10.4f}')

---
## 8. Engineering Insight

The Ridge model achieves strong predictive accuracy on the Fmax dataset. In a real post-silicon validation workflow, a model like this could:

- Predict Fmax at untested voltage/temperature corners, reducing the number of physical test runs by 40-60%.
- Flag silicon lots that are likely to fail timing at high temperature (outlier detection via residuals).
- Provide a fast first-pass estimate during bring-up, before full characterization is complete.

The Lasso model's feature selection capability is particularly useful here: if it drives `thermal_resistance` to zero, that tells us this feature is redundant given the other measurements — a useful insight for test plan optimization.

**Next:** Project 002 will tackle classification — predicting whether a chip passes or fails a specific timing test — using Logistic Regression.